In [1]:
import os
import lmdb
import json
import pickle
import rasterio
import datetime
import numpy as np
import matplotlib.pyplot as plt

from shapely.geometry import MultiPolygon
import geopandas as gpd
import pandas as pd

from tqdm import tqdm

fd_pdata = '/p/scratch/hai_dm4eo/liu_ch/data/WeakFT'
fd_lmdb = '/p/scratch/hai_ws4eo/liu_ch/data/WeakFT/weak_lmdbs'
fd_used_csv = '/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/data_preparation/weak_label_downloading/centroids_used'

S2C_MEAN = [1605.57504906, 1390.78157673, 1314.8729939, 1363.52445545, 1549.44374991, 2091.74883118, 2371.7172463, 2299.90463006, 2560.29504086, 830.06605044, 22.10351321, 2177.07172323, 1524.06546312]
S2C_STD = [786.78685367, 850.34818441, 875.06484736, 1138.84957046, 1122.17775652, 1161.59187054, 1274.39184232, 1248.42891965, 1345.52684884, 577.31607053, 51.15431158, 1336.09932639, 1136.53823676]

In [9]:
def get_complete_set(fdir, n=4):
    fdir = os.path.join(fdir, 'data')   
    ndirs = [int(d) for d in os.listdir(fdir) if '.' not in d]
    ndirs.sort()
    complete_set = []    # s1, s2, lc
    for nd in ndirs:
        fp = os.path.join(fdir, f'{nd:07d}')
        fsubs = os.listdir(fp)
        if len(fsubs) == n:
            complete_set.append(nd)
    return complete_set, len(ndirs)


def save_meta_info(fdir, complete_set, keys=['s1', 's2'], lc_keys=None):
    # Initialize lists to store the data
    names = {key: [] for key in keys}
    coords = []

    # Iterate through the complete_set
    for nd in complete_set:
        fp = os.path.join(fdir, 'data', f'{nd:07d}')
        name = {key: None for key in keys}
        
        # List the subdirectories and files
        fsubs = os.listdir(fp)
        
        # Find the s1 and s2 file names
        for fs in fsubs:
            if fs.endswith('.tif'):
                if lc_keys is None: continue
            else:
                name[fs] = os.listdir(os.path.join(fp, fs))[0]
        for k in name:
            assert name[k] is not None, f'Error: {k}'
            names[k].append(name[k])

    # Create a DataFrame
    df = pd.DataFrame({
        'id': complete_set,
        **names,
    })

    # Read the CSV file
    csv_path = os.path.join(fdir, 'Downloaded_file_info_lc.csv')
    csv_df = pd.read_csv(csv_path)

    # Fetch the corresponding coordinates
    coords, lons, lats = [], [], []
    for id in complete_set:
        coord_row = csv_df[csv_df['img_id'] == id].coords
        coords_ = json.loads(coord_row.iloc[0])
        coords.append(coords_)
        coords_ = np.array(coords_)
        center = [coords_[:, i].mean() for i in range(2)]
        lons.append(center[0])
        lats.append(center[1])

    # Add the coordinates to the DataFrame
    df['coords'] = coords
    df['lon'] = lons
    df['lat'] = lats

    # save the DataFrame
    df.to_csv(os.path.join(fdir, 'Complete_file_info.csv'), index=False)

    return df

In [10]:
def plot_points(shp, gdf_points, marker_size=.5):
    _, ax = plt.subplots(figsize=(10, 10))
    shp.plot(ax=ax, color='white', edgecolor='black')
    gdf_points.plot(ax=ax, color='red', markersize=marker_size)
    plt.show()
    return

def read_labels(gdf, fd_data):
    lcs = []
    for d in tqdm(gdf.id):
        fp = os.path.join(fd_data, f'{d:07d}', 'lc.tif')
        with rasterio.open(fp, 'r') as src:
            lcs.append(src.read().squeeze())
    lcs = np.dstack(lcs)
    lcs_unique = np.unique(lcs)
    print(f'shape of lcs: {lcs.shape}')
    print(f'unique values in lcs: {lcs_unique}')
    return lcs, lcs_unique

def get_label_proportions(lcs, lcs_unique):
    proportions = []
    for lc in lcs_unique:
        proportion = (lcs == lc).sum() / lcs.size
        proportions.append(proportion)
        print(f'Proportion of {lc:03d}: {proportion*100:.2f}%')
    return proportions

def show_image_example(fd_data, ind, img_names, img_bands=[3,2,1], lbl_name='lc', alpha=0.2, print_dtypes=False):
    n_img = len(img_names)
    # lbl
    fp_lbl = os.path.join(fd_data, f'{ind:07d}', f'{lbl_name}.tif')
    with rasterio.open(fp_lbl, 'r') as src:
        lbl = src.read().squeeze()
    # img
    imgs = {}
    for iname in img_names:
        fp_img = os.path.join(fd_data, f'{ind:07d}', f'{iname}.tif')
        with rasterio.open(fp_img, 'r') as src:
            img = src.read().squeeze()
            if print_dtypes:
                print(f'{iname}: {img.dtype}, {img.shape}')
        img = img[img_bands].transpose(1,2,0)
        img = (img - img.min()) / (img.max() - img.min())
        imgs[iname] = img
    # plot
    n = n_img*2 + 1
    plt.figure(figsize=(5*n, 5))
    plt.subplot(1, n, 1)
    plt.imshow(lbl, cmap='tab20', interpolation='none')
    plt.title('Label')
    for i, iname in enumerate(img_names):
        plt.subplot(1, n, 2+i)
        plt.imshow(imgs[iname])
        plt.title(iname)
    for i, iname in enumerate(img_names):
        plt.subplot(1, n, n_img+2+i)
        plt.imshow(imgs[iname])
        plt.imshow(lbl, cmap='tab20', alpha=alpha, interpolation='none')
    return

def create_lmdb(fp_data, fp_lmdb, gsize, ids, img_names, keys=None):
    lmdb_size = gsize*1024**3
    if keys is None:
        keys = img_names

    t0 = datetime.datetime.now().replace(microsecond=0)
    env = lmdb.open(fp_lmdb, map_size=lmdb_size)
    txn = env.begin(write=True)
    for di, d in tqdm(enumerate(ids), total=len(ids)):
        # read data
        fpd = os.path.join(fp_data, f'{d:07d}')
        for iname, ikey in zip(img_names, keys):
            fp = os.path.join(fpd, f'{iname}.tif')
            with rasterio.open(fp) as src:
                img = src.read().squeeze()
            if iname == 'lc' or iname == 'bc':
                assert img.shape == (264, 264), f'Error: {iname}-{img.shape}-{fp}'
            else:
                assert img.shape[1:] == (264, 264), f'Error: {iname}-{img.shape}-{fp}'
            # save data
            obj = (img.tobytes(), img.shape, d)    
            key = f'{di}_{ikey}' 
            txn.put(key.encode(), pickle.dumps(obj))
    # record basic key names
    txn.put('keys'.encode(), pickle.dumps(keys))
    txn.commit()
    env.sync()
    env.close()
    t1 = datetime.datetime.now().replace(microsecond=0)
    print(f'Read data into {fp_lmdb}|Elapsed time: {t1-t0}')
    return

# Iran

In [ ]:
# shapefile
iran_shapefile = gpd.read_file('/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/data_downloading/Iran_shp/irn_admbnda_adm0_unhcr_20190514.shp')
# data points
fdir_iran = os.path.join(fd_pdata, 'Iran')
fdir_iran_data = os.path.join(fdir_iran, 'data')
fdir_iran_csv  = os.path.join(fdir_iran, 'Downloaded_file_info_lc.csv')
points_iran = pd.read_csv(fdir_iran_csv, header=0)
# Convert the DataFrame to a GeoDataFrame
gdf_coords_iran = gpd.GeoDataFrame(points_iran, geometry=gpd.points_from_xy(points_iran.longitude, points_iran.latitude))
len(gdf_coords_iran)

## plot spots - 5480

In [ ]:
plot_points(iran_shapefile, gdf_coords_iran, marker_size=1)

## class proportion

In [ ]:
lcs, lcs_unique = read_labels(gdf_coords_iran, fdir_iran_data)

In [ ]:
lcs_unique = np.unique(lcs)
lcs_proportions = get_label_proportions(lcs, lcs_unique)

## show images

In [ ]:
i = 0
ind = gdf_coords_iran.id.iloc[i]
show_image_example(fdir_iran_data, ind, img_names=['s2_l1c'], img_bands=[3,2,1], lbl_name='lc', alpha=0.25)

## shuffle the indexes and create lmdb

In [ ]:
fp_new_csv_iran = os.path.join(fd_used_csv, 'center_coords_iran.csv')
# shuffle indexes
shuffled_indexes = np.random.permutation(len(gdf_coords_iran))
new_gdf_coords_iran = gdf_coords_iran.iloc[shuffled_indexes]
new_gdf_coords_iran.reset_index(drop=True, inplace=True)
new_gdf_coords_iran.loc[:, ['id', 'longitude', 'latitude', 's2_imageid', 'coords', 'n_nodata']].to_csv(fp_new_csv_iran, index=False)
new_gdf_coords_iran

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'iran.lmdb')
gsize = 10
img_names = ['s2_l1c', 'lc']

create_lmdb(fdir_iran_data, fp_lmdb, gsize, new_gdf_coords_iran.id, img_names)

In [ ]:
# check saved lmdb file
i = 2
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    img_obj = pickle.loads(txn.get((f'{i}_s2_l1c').encode()))
    lbl_obj = pickle.loads(txn.get((f'{i}_lc').encode()))
    img = np.frombuffer(img_obj[0], dtype=np.uint16).reshape(img_obj[1])
    lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
    print(img_obj[-1], lbl_obj[-1], img.shape, lbl.shape, n_items, int((n_items-1)//2), keys)
    plt.figure(figsize=(15, 5))
    plt.subplot(131)
    plt.imshow(lbl, cmap='tab20', interpolation='none')
    plt.subplot(132)
    im = img[[3,2,1]].transpose(1,2,0)
    im = (im - np.min(im)) / (np.max(im) - np.min(im))

    # # ssl4eo-s12_0k_251k_s2c_norm.lmdb (used by NS_Pr project)
    # im = []
    # for i in [3,2,1]:
    #     im_ = img[i].astype(np.float32)
    #     vmin = S2C_MEAN[i] - 2*S2C_STD[i]
    #     vmax = S2C_MEAN[i] + 2*S2C_STD[i]
    #     im_ = ((im_-vmin) / (vmax-vmin))*255
    #     im_ = im_.astype(np.uint8)
    #     im.append(im_)
    # im = np.stack(im, axis=2)
    
    # # ssl4eo-s12_0k_251k.lmdb
    # im = img[[3,2,1]].transpose(1,2,0)
    # im = im/10000 # (im - np.min(im)) / (np.max(im) - np.min(im))

    plt.imshow(im)
    plt.subplot(133)
    plt.imshow(im)
    plt.imshow(lbl, cmap='tab20', alpha=0.2, interpolation='none')

In [ ]:
show_image_example(fdir_iran_data, img_obj[-1], img_names=['s2_l1c'], img_bands=[3,2,1], lbl_name='lc', alpha=0.2)

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'iran.lmdb')
labels = []
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    for i in range(int((n_items-1)//2)):
        lbl_obj = pickle.loads(txn.get((f'{i}_lc').encode()))
        lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
        labels.append(lbl)
labels = np.dstack(labels)
n_items, keys, labels.shape, np.unique(labels)

# WorldCover

In [ ]:
# shapefile
world_shapefile = gpd.read_file('world_shp/world-administrative-boundaries.shp')
# data points
fdir_wc_data = {}
fdir_wcv1 = os.path.join(fd_pdata,'worldcover_v100')
fdir_wcv1_data = os.path.join(fdir_wcv1, 'data')
fdir_wcv1_csv  = os.path.join(fdir_wcv1, 'Downloaded_file_info_lc.csv')
points_wcv1 = pd.read_csv(fdir_wcv1_csv, header=0)
fdir_wc_data['worldcover_v100'] = fdir_wcv1_data
fdir_wcv2 = os.path.join(fd_pdata,'worldcover_v200')
fdir_wcv2_data = os.path.join(fdir_wcv2, 'data')
fdir_wcv2_csv  = os.path.join(fdir_wcv2, 'Downloaded_file_info_lc.csv')
points_wcv2 = pd.read_csv(fdir_wcv2_csv, header=0)
fdir_wc_data['worldcover_v200'] = fdir_wcv2_data
# Convert the DataFrame to a GeoDataFrame
dfs = {}
dfs['worldcover_v100'] = gpd.GeoDataFrame(points_wcv1, geometry=gpd.points_from_xy(points_wcv1.longitude, points_wcv1.latitude))
dfs['worldcover_v200'] = gpd.GeoDataFrame(points_wcv2, geometry=gpd.points_from_xy(points_wcv2.longitude, points_wcv2.latitude))
for k in dfs:
    print(f'{k}: {len(dfs[k])}')

In [ ]:
dfs[k]

## plot the overlap points - 44087

In [ ]:
d1 = dfs['worldcover_v100'].id
d2 = dfs['worldcover_v200'].id
overlap = d1[d1.isin(d2)]
gdf_overlap_points = dfs['worldcover_v100'].iloc[overlap.index]
fig, ax = plt.subplots(figsize=(10, 10))
world_shapefile.plot(ax=ax, color='white', edgecolor='black')
gdf_overlap_points.plot(ax=ax, color='red', markersize=.5)
plt.show()
len(gdf_overlap_points)

In [ ]:
gdf_overlap_points

## shuffle the indexes and create lmdb

In [ ]:
fp_new_csv_wv = os.path.join(fd_used_csv, 'center_coords_overlap_worldcover.csv')
shuffled_indexes = np.random.permutation(len(gdf_overlap_points))
new_gdf_overlap_points = gdf_overlap_points.iloc[shuffled_indexes]
new_gdf_overlap_points.reset_index(drop=True, inplace=True)
overlap_points = new_gdf_overlap_points.loc[:, ['id', 'longitude', 'latitude', 's2_imageid', 'coords']]
overlap_points.to_csv(fp_new_csv_wv, index=False)
new_gdf_overlap_points

In [ ]:
fp_new_csv_wv = os.path.join(fd_used_csv, 'center_coords_overlap_worldcover.csv')
new_gdf_overlap_points = pd.read_csv(fp_new_csv_wv)
new_gdf_overlap_points

In [ ]:
gsize = 147
img_names = ['s2_l1c', 's2_l2a', 'lc']

for v in [100, 200]:
    fp_lmdb = os.path.join(fd_lmdb, f'worldcover_v{v}.lmdb')
    create_lmdb(fdir_wc_data[f'worldcover_v{v}'], fp_lmdb, gsize, new_gdf_overlap_points.id, img_names)

In [ ]:
# check saved lmdb file
i = 2
v = 200
fp_lmdb = os.path.join(fd_lmdb, f'worldcover_v{v}.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    img_obj = pickle.loads(txn.get((f'{i}_s2_l1c').encode()))
    imga_obj = pickle.loads(txn.get((f'{i}_s2_l2a').encode()))
    lbl_obj = pickle.loads(txn.get((f'{i}_lc').encode()))
    img = np.frombuffer(img_obj[0], dtype=np.uint16).reshape(img_obj[1])
    imga = np.frombuffer(imga_obj[0], dtype=np.uint16).reshape(imga_obj[1])
    lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
    print(img_obj[-1], lbl_obj[-1], img.shape, lbl.shape, n_items, int((n_items-1)//2), keys)
    plt.figure(figsize=(25, 5))
    plt.subplot(151)
    plt.imshow(lbl, cmap='tab20', interpolation='none')
    plt.subplot(152)
    im = img[[3,2,1]].transpose(1,2,0)
    im = (im - np.min(im)) / (np.max(im) - np.min(im))
    plt.imshow(im)
    plt.subplot(153)
    ima = imga[[3,2,1]].transpose(1,2,0)
    ima = (ima - np.min(ima)) / (np.max(ima) - np.min(ima))
    plt.imshow(ima)
    plt.subplot(154)
    plt.imshow(im)
    plt.imshow(lbl, cmap='tab20', alpha=0.2, interpolation='none')
    plt.subplot(155)
    plt.imshow(ima)
    plt.imshow(lbl, cmap='tab20', alpha=0.2, interpolation='none')

In [ ]:
(n_items-1)/3

In [ ]:
show_image_example(fdir_wc_data[f'worldcover_v{v}'], img_obj[-1], img_names=['s2_l1c', 's2_l2a'], img_bands=[3,2,1], lbl_name='lc', alpha=0.2)

## class proportions

In [ ]:
for v in [100, 200]:
    fp_lmdb = os.path.join(fd_lmdb, f'worldcover_v{v}.lmdb')
    labels = []
    env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
    with env.begin(write=False) as txn:
        n_items = txn.stat()['entries']
        keys = pickle.loads(txn.get('keys'.encode()))
        for i in range(int((n_items-1)//3)):
            lbl_obj = pickle.loads(txn.get((f'{i}_lc').encode()))
            lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
            labels.append(lbl)
    labels = np.dstack(labels)
    uni_vs = np.unique(labels)
    print(f'v{v}', n_items, keys, labels.shape, uni_vs)
    for uni_v in uni_vs:
        na = np.sum(labels==uni_v)
        print(f'v{v} {uni_v}: {na}, {na/labels.size*100:.2f}%')

In [ ]:
dict_tmp = {
    10: 638668598+686315701,
    20: 216032085+198665229,
    30: 606015485+630746085,
    40: 742009258+734954678,
    50: 138763294+152008727,
    60: 398807687+336827437,
    70: 99225+107139,
    80: 314401592+314841684,
    90: 16202326+16868763,
    95: 1402079+1211999,
    100: 216227+70414
}
sum_all = sum(dict_tmp.values())
for k in dict_tmp:
    print(f'{k}: {dict_tmp[k]}, {dict_tmp[k]/sum_all*100:.4f}%')

In [ ]:
lbl = labels.copy()
lbl[lbl>90] = lbl[lbl>90]+10
lbl = np.floor(lbl/10).astype(np.uint8)
lbl = lbl - 1
np.unique(lbl)

In [ ]:
lbl_obj[-1]

# GHSL

In [ ]:
# shapefile
world_shapefile = gpd.read_file('world_shp/world-administrative-boundaries.shp')
# data points
fdir_ghsl_data = {}
fdir_ghsl = os.path.join(fd_pdata,'ghsl_s2c_only')
fdir_ghsl_data = os.path.join(fdir_ghsl, 'data')
fdir_ghsl_csv  = os.path.join(fdir_ghsl, 'Downloaded_file_info_lc.csv')
points_ghsl = pd.read_csv(fdir_ghsl_csv, header=0)

# Convert the DataFrame to a GeoDataFrame
df_ghsl = gpd.GeoDataFrame(points_ghsl, geometry=gpd.points_from_xy(points_ghsl.longitude, points_ghsl.latitude))
len(df_ghsl)

In [ ]:
df_ghsl

In [ ]:
plot_points(world_shapefile, df_ghsl, marker_size=1)

In [ ]:
fp_new_csv_ghsl = os.path.join(fd_used_csv, 'center_coords_ghsl.csv')
# shuffle indexes
shuffled_indexes = np.random.permutation(len(df_ghsl))
new_gdf_coords_ghsl = df_ghsl.iloc[shuffled_indexes]
new_gdf_coords_ghsl.reset_index(drop=True, inplace=True)
new_gdf_coords_ghsl.loc[:, ['id', 'longitude', 'latitude', 's2_imageid', 'coords', 'n_nodata']].to_csv(fp_new_csv_ghsl, index=False)
new_gdf_coords_ghsl

In [ ]:
# read all the label masks
lbls = []
for i in tqdm(range(len(new_gdf_coords_ghsl)), total=len(new_gdf_coords_ghsl)):
    ind = new_gdf_coords_ghsl.id.iloc[i]
    fp = os.path.join(fdir_ghsl_data, f'{ind:07d}', 'bc.tif')
    with rasterio.open(fp, 'r') as src:
        lbl = src.read().squeeze()
        lbls.append(lbl)
lbls = np.dstack(lbls)
tmp = lbls>10
n_data = np.sum(tmp, axis=(0,1))
np.sum(n_data==0)

In [ ]:
np.sum(n_data>=264*10)

In [ ]:
select_gdf_coords_ghsl = new_gdf_coords_ghsl.iloc[n_data>=264*10]
fp_new_csv_ghsl_select = os.path.join(fd_used_csv, 'center_coords_ghsl_select.csv')
select_gdf_coords_ghsl.to_csv(fp_new_csv_ghsl_select, index=False)
select_gdf_coords_ghsl

In [ ]:
lbls_sel = []
for i in tqdm(range(len(select_gdf_coords_ghsl)), total=len(select_gdf_coords_ghsl)):
    ind = select_gdf_coords_ghsl.id.iloc[i]
    fp = os.path.join(fdir_ghsl_data, f'{ind:07d}', 'bc.tif')
    with rasterio.open(fp, 'r') as src:
        lbl = src.read().squeeze()
        lbls_sel.append(lbl)
lbls_sel = np.dstack(lbls_sel)
tmp = lbls_sel>10
n_data_sel = np.sum(tmp, axis=(0,1))
np.sum(n_data_sel<264*10)

In [ ]:
plot_points(world_shapefile, select_gdf_coords_ghsl, marker_size=1)

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'ghsl_select.lmdb')
gsize = 17
img_names = ['s2_l1c', 'bc']

create_lmdb(fdir_ghsl_data, fp_lmdb, gsize, select_gdf_coords_ghsl.id, img_names)

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'ghsl_select.lmdb')
labels = []
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    for i in range(int((n_items-1)//2)):
        lbl_obj = pickle.loads(txn.get((f'{i}_bc').encode()))
        lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
        labels.append(lbl)
labels = np.dstack(labels)
uni_vs = np.unique(labels)
print(n_items, keys, labels.shape, uni_vs)
for uni_v in uni_vs:
    na = np.sum(labels==uni_v)
    print(f'{uni_v}: {na}, {na/labels.size*100:.2f}%, {na/labels.size*100:.4f}%')

In [ ]:
sum_all = {}
for i, uni_vs in enumerate([[1,2,3,4],[11,12,13,14,15],[21,22,23,24,25]]):
    tmp_ = 0
    for v in uni_vs:
        tmp_ += np.sum(labels==v)
    sum_all[i] = tmp_
    print(f'{i}: {tmp_}, {tmp_/labels.size*100:.2f}%')

# dynamicEarthNet

In [20]:
fp = '/p/project1/hai_ws4eo/liu_ch/data/WeakFT/exact/dynamicearthnet_seasonal.lmdb'
env = lmdb.open(fp, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
lbls = []
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    n_ind = int(n_items/6)
    print(n_items, n_ind)
    for ind in range(n_ind):
        for i in range(6):
            key = f'{ind}_{i}'
            data = pickle.loads(txn.get(key.encode()))
            lbl = np.frombuffer(data[2], dtype=np.uint8).reshape(data[3])
            lbls.append(lbl)
lbls = np.dstack(lbls)
lbls.shape, np.unique(lbls)

28800 4800


((256, 256, 28800), array([0, 1, 2, 3, 4, 5, 6], dtype=uint8))

In [21]:
for i in range(7):
    print(f'{i}: {np.sum(lbls==i)}, {np.sum(lbls==i)/lbls.size*100:.2f}%')

0: 135129241, 7.16%
1: 180235348, 9.55%
2: 858550135, 45.49%
3: 13530906, 0.72%
4: 529630955, 28.06%
5: 151660354, 8.04%
6: 18699861, 0.99%


In [22]:
fp = '/p/project1/hai_ws4eo/liu_ch/data/WeakFT/exact/openearthmap_train.lmdb'
env = lmdb.open(fp, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
lbls = []
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    print(n_items)
    for ind in range(n_items):
        data = pickle.loads(txn.get(str(ind).encode()))
        lbl = np.frombuffer(data[2], dtype=np.uint8).reshape(data[3])
        lbls.append(lbl)
lbls = np.dstack(lbls)
unis = np.unique(lbls)
print(lbls.shape, unis)
for i in unis:
    print(f'{i}: {np.sum(lbls==i)}, {np.sum(lbls==i)/lbls.size*100:.2f}%')

25080
(320, 320, 25080) [0 1 2 3 4 5 6 7 8]
0: 3371772, 0.13%
1: 38837379, 1.51%
2: 582265650, 22.67%
3: 403890406, 15.73%
4: 170042453, 6.62%
5: 521097910, 20.29%
6: 85401430, 3.33%
7: 366090856, 14.25%
8: 397194144, 15.47%


# NLCD

In [ ]:
# create continental US shapefile
world_shapefile = gpd.read_file('/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/data_downloading/world_shp/world-administrative-boundaries.shp')
# Create an empty list to store individual polygons
us_shapefile = world_shapefile[world_shapefile['name'] == 'United States of America']
split_polygons = []
min_longitude, max_longitude = -128, -50

# Iterate over each geometry in the GeoDataFrame
for geom in us_shapefile.geometry:
    if isinstance(geom, MultiPolygon):
        # If the geometry is a MultiPolygon, iterate over individual polygons
        for poly in geom.geoms:
            # Check the bounds of the polygon (min_x, min_y, max_x, max_y)
            if poly.bounds[0] >= min_longitude and poly.bounds[2] <= max_longitude:
                split_polygons.append(poly)
    else:
        # If the geometry is already a Polygon, check its bounds
        if geom.bounds[0] >= min_longitude and geom.bounds[2] <= max_longitude:
            split_polygons.append(geom)

# Create a new GeoDataFrame with the split polygons
split_gdf = gpd.GeoDataFrame(geometry=split_polygons)
print(split_gdf.shape)

In [ ]:
# data points
fdir_nlcd = '/p/scratch/hai_ws4eo/liu_ch/data/WeakFT/nlcd/'
fdir_nlcd_data = os.path.join(fdir_nlcd, 'data')
fdir_nlcd_csv  = os.path.join(fdir_nlcd, 'Mask_success_center_coords_usa_nlcd.csv')
points_nlcd = pd.read_csv(fdir_nlcd_csv, header=0)
# Convert the DataFrame to a GeoDataFrame
gdf_coords_nlcd = gpd.GeoDataFrame(points_nlcd, geometry=gpd.points_from_xy(points_nlcd.longitude, points_nlcd.latitude))
len(gdf_coords_nlcd)

## plot spots - 18735

In [ ]:
# plot the points
fig, ax = plt.subplots(figsize=(10, 10))
split_gdf.plot(ax=ax, color='white', edgecolor='black')
gdf_coords_nlcd.plot(ax=ax, color='red', markersize=.5)

In [ ]:
gdf_coords_nlcd

## shuffle the indexes and create lmdb

In [ ]:
fp_new_csv_nlcd = os.path.join(fd_used_csv, 'center_coords_nlcd.csv')
shuffled_indexes = np.random.permutation(len(gdf_coords_nlcd))
new_gdf_coords_nlcd = gdf_coords_nlcd.iloc[shuffled_indexes]
new_gdf_coords_nlcd.reset_index(drop=True, inplace=True)
new_points_nlcd = new_gdf_coords_nlcd.loc[:, ['id', 'longitude', 'latitude', 'coords', 'n_nodata_lu']]
new_points_nlcd.to_csv(fp_new_csv_nlcd, index=False)
new_gdf_coords_nlcd

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'nlcd.lmdb')
gsize = 75
img_names = ['toa', 'sr', 'nlcd_landcover_2019', 'nlcd_impervious_2019']
keys = ['toa', 'sr', 'lc', 'imp']

create_lmdb(fdir_nlcd_data, fp_lmdb, gsize, new_gdf_coords_nlcd.id, img_names, keys)

In [ ]:
# check saved lmdb file
i = 500
lbl_k = 'lc'
fp_lmdb = os.path.join(fd_lmdb, 'nlcd.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    img_obj = pickle.loads(txn.get((f'{i}_toa').encode()))
    imgs_obj = pickle.loads(txn.get((f'{i}_sr').encode()))
    lbl_obj = pickle.loads(txn.get((f'{i}_{lbl_k}').encode()))
    img = np.frombuffer(img_obj[0], dtype=np.float32).reshape(img_obj[1])
    imgs = np.frombuffer(imgs_obj[0], dtype=np.uint16).reshape(imgs_obj[1])
    lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
    print(img_obj[-1], lbl_obj[-1], img.shape, lbl.shape, n_items, int((n_items-1)//2), keys)
    plt.figure(figsize=(25, 5))
    plt.subplot(151)
    plt.imshow(lbl, cmap='tab20', interpolation='none')
    plt.subplot(152)
    im = img[[3,2,1]].transpose(1,2,0)
    im = (im - np.min(im)) / (np.max(im) - np.min(im))
    plt.imshow(im)
    plt.subplot(153)
    ims = imgs[[3,2,1]].transpose(1,2,0)
    ims = (ims - np.min(ims)) / (np.max(ims) - np.min(ims))
    plt.imshow(ims)
    plt.subplot(154)
    plt.imshow(im)
    plt.imshow(lbl, cmap='tab20', alpha=0.4, interpolation='none')
    plt.subplot(155)
    plt.imshow(ims)
    plt.imshow(lbl, cmap='tab20', alpha=0.4, interpolation='none')

## calculate mean and std

In [ ]:
# mean and std
n_img = 4
fp_lmdb = os.path.join(fd_lmdb, 'nlcd.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
meant, stdt = [], []
means, stds = [], []
with env.begin(write=False) as txn:
    n_items = int(txn.stat()['entries']//n_img)
    for i in range(n_items):
        img_obj = pickle.loads(txn.get((f'{i}_toa').encode()))
        imgs_obj = pickle.loads(txn.get((f'{i}_sr').encode()))
        img = np.frombuffer(img_obj[0], dtype=np.float32).reshape(img_obj[1])
        imgs = np.frombuffer(imgs_obj[0], dtype=np.uint16).reshape(imgs_obj[1]).astype(np.float32)
        meant.append(img.mean(axis=(1,2)))
        stdt.append(img.std(axis=(1,2)))
        means.append(imgs.mean(axis=(1,2)))
        stds.append(imgs.std(axis=(1,2)))
meant, stdt = np.stack(meant, axis=0), np.stack(stdt, axis=0)
means, stds = np.stack(means, axis=0), np.stack(stds, axis=0)
# overall means and stds
meant_overall_nlcd = meant.mean(axis=0)
stdt_overall_nlcd = np.sqrt(np.mean(stdt**2 + meant**2, axis=0) - meant_overall_nlcd**2)
means_overall_nlcd = means.mean(axis=0)
stds_overall_nlcd = np.sqrt(np.mean(stds**2 + means**2, axis=0) - means_overall_nlcd**2)
print(f'TOA: mean={meant_overall_nlcd}, std={stdt_overall_nlcd}')
print(f'SR: mean={means_overall_nlcd}, std={stds_overall_nlcd}')

## class proportions

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'nlcd.lmdb')
key_list = ['lc', 'imp']
lbls = {k:[] for k in key_list}
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    print(keys, n_items)
    for i in range(int((n_items-1)//4)):
        for k in key_list:
            lbl_obj = pickle.loads(txn.get((f'{i}_{k}').encode()))
            lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
            lbls[k].append(lbl)

for k in key_list:
    lbls_ = np.dstack(lbls[k])
    uni_vs = np.unique(lbls_)
    print(f'key={k}', lbls_.shape, uni_vs)
    for uni_v in uni_vs:
        na = np.sum(lbls_==uni_v)
        print(f'{uni_v}: {na}, {na/lbls_.size*100:.2f}%')

In [ ]:
tmp = [1488335, 5384824, 27325112, 6489070, 16841852, 156, 11817, 8937, 88018, 335967,]
len(tmp)

In [ ]:
import numpy as np  
np.sum(tmp[:4]/np.sum(tmp)), np.sum(tmp[4:7]/np.sum(tmp)), np.sum(tmp[7:]/np.sum(tmp))

# USFS

In [ ]:
# data points
fdir_usfs = os.path.join(fd_pdata, 'usfs')
fdir_usfs_data = os.path.join(fdir_usfs, 'data')
fdir_usfs_csv  = os.path.join(fdir_usfs, 'Mask_success_center_coords_usa_usfs.csv')
points_usfs = pd.read_csv(fdir_usfs_csv, header=0)
# Convert the DataFrame to a GeoDataFrame
gdf_coords_usfs = gpd.GeoDataFrame(points_usfs, geometry=gpd.points_from_xy(points_usfs.longitude, points_usfs.latitude))
len(gdf_coords_usfs)

## plot spots - 18582

In [ ]:
# plot the points
fig, ax = plt.subplots(figsize=(10, 10))
split_gdf.plot(ax=ax, color='white', edgecolor='black')
gdf_coords_usfs.plot(ax=ax, color='red', markersize=.5)

In [ ]:
gdf_coords_usfs

In [ ]:
(gdf_coords_usfs.n_nodata_lc!=0).sum(), (gdf_coords_usfs.n_nodata_lu!=0).sum()

## shuffle the indexes and create lmdb

In [ ]:
fp_new_csv_usfs = os.path.join(fd_used_csv, 'center_coords_usfs.csv')
shuffled_indexes = np.random.permutation(len(gdf_coords_usfs))
new_gdf_coords_usfs = gdf_coords_usfs.iloc[shuffled_indexes]
new_gdf_coords_usfs.reset_index(drop=True, inplace=True)
new_points_usfs = new_gdf_coords_usfs.loc[:, ['id', 'longitude', 'latitude', 'coords']]
new_points_usfs.to_csv(fp_new_csv_usfs, index=False)
new_gdf_coords_usfs

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'usfs.lmdb')
gsize = 75
img_names = ['toa', 'sr', 'usfs_landcover_2023', 'usfs_landuse_2023']
keys = ['toa', 'sr', 'lc', 'lu']

create_lmdb(fdir_usfs_data, fp_lmdb, gsize, new_gdf_coords_usfs.id, img_names, keys)

In [ ]:
# check saved lmdb file
i = 200
lbl_k = 'lc'
fp_lmdb = os.path.join(fd_lmdb, 'usfs.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    img_obj = pickle.loads(txn.get((f'{i}_toa').encode()))
    imgs_obj = pickle.loads(txn.get((f'{i}_sr').encode()))
    lbl_obj = pickle.loads(txn.get((f'{i}_{lbl_k}').encode()))
    img = np.frombuffer(img_obj[0], dtype=np.float32).reshape(img_obj[1])
    imgs = np.frombuffer(imgs_obj[0], dtype=np.uint16).reshape(imgs_obj[1])
    lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
    print(img_obj[-1], lbl_obj[-1], img.shape, lbl.shape, n_items, int((n_items-1)//4), keys)
    plt.figure(figsize=(25, 5))
    plt.subplot(151)
    plt.imshow(lbl, cmap='tab20', interpolation='none')
    plt.subplot(152)
    im = img[[3,2,1]].transpose(1,2,0)
    im = (im - np.min(im)) / (np.max(im) - np.min(im))
    plt.imshow(im)
    plt.subplot(153)
    ims = imgs[[3,2,1]].transpose(1,2,0)
    ims = (ims - np.min(ims)) / (np.max(ims) - np.min(ims))
    plt.imshow(ims)
    plt.subplot(154)
    plt.imshow(im)
    plt.imshow(lbl, cmap='tab20', alpha=0.4, interpolation='none')
    plt.subplot(155)
    plt.imshow(ims)
    plt.imshow(lbl, cmap='tab20', alpha=0.4, interpolation='none')

In [ ]:
# lbl_k = 'lu'
lbl_name = 'usfs_landcover_2023' if lbl_k == 'lc' else 'usfs_landuse_2023'
show_image_example(fdir_usfs_data, img_obj[-1], img_names=['sr'], img_bands=[3,2,1], lbl_name=lbl_name, alpha=0.5, print_dtypes=True)  # img_obj[-1]

## calculate mean and std

In [ ]:
# mean and std
n_img = 4
fp_lmdb = os.path.join(fd_lmdb, 'usfs.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
meant, stdt = [], []
means, stds = [], []
with env.begin(write=False) as txn:
    n_items = int(txn.stat()['entries']//n_img)
    for i in range(n_items):
        img_obj = pickle.loads(txn.get((f'{i}_toa').encode()))
        imgs_obj = pickle.loads(txn.get((f'{i}_sr').encode()))
        img = np.frombuffer(img_obj[0], dtype=np.float32).reshape(img_obj[1])
        imgs = np.frombuffer(imgs_obj[0], dtype=np.uint16).reshape(imgs_obj[1]).astype(np.float32)
        meant.append(img.mean(axis=(1,2)))
        stdt.append(img.std(axis=(1,2)))
        means.append(imgs.mean(axis=(1,2)))
        stds.append(imgs.std(axis=(1,2)))
meant, stdt = np.stack(meant, axis=0), np.stack(stdt, axis=0)
means, stds = np.stack(means, axis=0), np.stack(stds, axis=0)
# overall means and stds
meant_overall_usfs = meant.mean(axis=0)
stdt_overall_usfs = np.sqrt(np.mean(stdt**2 + meant**2, axis=0) - meant_overall_usfs**2)
means_overall_usfs = means.mean(axis=0)
stds_overall_usfs = np.sqrt(np.mean(stds**2 + means**2, axis=0) - means_overall_usfs**2)
print(n_items, meant.shape, stdt.shape, means.shape, stds.shape)
print(f'TOA: mean={meant_overall_usfs}, std={stdt_overall_usfs}')
print(f'SR: mean={means_overall_usfs}, std={stds_overall_usfs}')

In [ ]:
img.min(axis=(1,2)), img.max(axis=(1,2))

## class proportions

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'usfs.lmdb')
key_list = ['lc', 'lu']
lbls = {k:[] for k in key_list}
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    print(keys, n_items)
    for i in range(int((n_items-1)//4)):
        for k in key_list:
            lbl_obj = pickle.loads(txn.get((f'{i}_{k}').encode()))
            lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
            lbls[k].append(lbl)

for k in key_list:
    lbls_ = np.dstack(lbls[k])
    uni_vs = np.unique(lbls_)
    print(f'key={k}', lbls_.shape, uni_vs)
    for uni_v in uni_vs:
        na = np.sum(lbls_==uni_v)
        print(f'{uni_v}: {na}, {na/lbls_.size*100:.2f}%')

In [ ]:
23205/(264*264*18595)*100


In [ ]:
lbls_array = {}
for k in lbls:
    lbls_array[k] = np.dstack(lbls[k])
    print(k, lbls_array[k].shape)
tmp = lbls_array['lc'][lbls_array['lu']==1]
uni_tmp = np.unique(tmp)
print(tmp.shape, uni_tmp)
for uni_t in uni_tmp:
    na = np.sum(tmp==uni_t)
    print(f'{uni_t}: {na}, {na/tmp.size*100:.2f}%')

# sbtn

In [ ]:
# data points
fdir_sbtn = os.path.join(fd_pdata, 'sbtn')
fdir_sbtn_data = os.path.join(fdir_sbtn, 'data')
fdir_sbtn_csv  = os.path.join(fdir_sbtn, 'Mask_success_center_coords_usa_sbtn.csv')
points_sbtn = pd.read_csv(fdir_sbtn_csv, header=0)
# Convert the DataFrame to a GeoDataFrame
gdf_coords_sbtn = gpd.GeoDataFrame(points_sbtn, geometry=gpd.points_from_xy(points_sbtn.longitude, points_sbtn.latitude))
len(gdf_coords_sbtn)

In [ ]:
gdf_coords_sbtn

## plot spots - 22687

In [ ]:
# plot the points
fig, ax = plt.subplots(figsize=(10, 10))
world_shapefile.plot(ax=ax, color='white', edgecolor='black')
gdf_coords_sbtn.plot(ax=ax, color='red', markersize=.5)

## shuffle the indexes and create lmdb

In [ ]:
fp_new_csv_sbtn = os.path.join(fd_used_csv, 'center_coords_sbtn.csv')
shuffled_indexes = np.random.permutation(len(gdf_coords_sbtn))
new_gdf_coords_sbtn = gdf_coords_sbtn.iloc[shuffled_indexes]
new_gdf_coords_sbtn.reset_index(drop=True, inplace=True)
new_points_sbtn = new_gdf_coords_sbtn.loc[:, ['id', 'longitude', 'latitude', 'coords']]
new_points_sbtn.to_csv(fp_new_csv_sbtn, index=False)
new_gdf_coords_sbtn

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'sbtn.lmdb')
gsize = 90
img_names = ['toa', 'sr', 'sbtn_landcover_2020']
keys = ['toa', 'sr', 'lc']

create_lmdb(fdir_sbtn_data, fp_lmdb, gsize, new_gdf_coords_sbtn.id, img_names, keys)

In [ ]:
# check saved lmdb file
i = 200
lbl_k = 'lc'
fp_lmdb = os.path.join(fd_lmdb, 'sbtn.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    img_obj = pickle.loads(txn.get((f'{i}_toa').encode()))
    imgs_obj = pickle.loads(txn.get((f'{i}_sr').encode()))
    lbl_obj = pickle.loads(txn.get((f'{i}_{lbl_k}').encode()))
    img = np.frombuffer(img_obj[0], dtype=np.float32).reshape(img_obj[1])
    imgs = np.frombuffer(imgs_obj[0], dtype=np.uint16).reshape(imgs_obj[1])
    lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
    print(img_obj[-1], lbl_obj[-1], img.shape, lbl.shape, n_items, int((n_items-1)//2), keys)
    plt.figure(figsize=(25, 5))
    plt.subplot(151)
    plt.imshow(lbl, cmap='tab20', interpolation='none')
    plt.subplot(152)
    im = img[[3,2,1]].transpose(1,2,0)
    im = (im - np.min(im)) / (np.max(im) - np.min(im))
    plt.imshow(im)
    plt.subplot(153)
    ims = imgs[[3,2,1]].transpose(1,2,0)
    ims = (ims - np.min(ims)) / (np.max(ims) - np.min(ims))
    plt.imshow(ims)
    plt.subplot(154)
    plt.imshow(im)
    plt.imshow(lbl, cmap='tab20', alpha=0.4, interpolation='none')
    plt.subplot(155)
    plt.imshow(ims)
    plt.imshow(lbl, cmap='tab20', alpha=0.4, interpolation='none')

In [ ]:
show_image_example(fdir_sbtn_data, img_obj[-1], img_names=['sr'], img_bands=[3,2,1], lbl_name='sbtn_landcover_2020', alpha=0.2, print_dtypes=True)  # img_obj[-1]

## calculate mean and std

In [ ]:
# check saved lmdb file
fp_lmdb = os.path.join(fd_lmdb, 'sbtn.lmdb')
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
meant, stdt = [], []
means, stds = [], []
with env.begin(write=False) as txn:
    n_items = int(txn.stat()['entries']//3)
    for i in range(n_items):
        img_obj = pickle.loads(txn.get((f'{i}_toa').encode()))
        imgs_obj = pickle.loads(txn.get((f'{i}_sr').encode()))
        img = np.frombuffer(img_obj[0], dtype=np.float32).reshape(img_obj[1])
        imgs = np.frombuffer(imgs_obj[0], dtype=np.uint16).reshape(imgs_obj[1]).astype(np.float32)
        meant.append(img.mean(axis=(1,2)))
        stdt.append(img.std(axis=(1,2)))
        means.append(imgs.mean(axis=(1,2)))
        stds.append(imgs.std(axis=(1,2)))
meant, stdt = np.stack(meant, axis=0), np.stack(stdt, axis=0)
means, stds = np.stack(means, axis=0), np.stack(stds, axis=0)
# overall means and stds
meant_overall_sbtn = meant.mean(axis=0)
stdt_overall_sbtn = np.sqrt(np.mean(stdt**2 + meant**2, axis=0) - meant_overall_sbtn**2)
means_overall_sbtn = means.mean(axis=0)
stds_overall_sbtn = np.sqrt(np.mean(stds**2 + means**2, axis=0) - means_overall_sbtn**2)
print(f'TOA: mean={meant_overall_sbtn}, std={stdt_overall_sbtn}')
print(f'SR: mean={means_overall_sbtn}, std={stds_overall_sbtn}')
print(f"TOA: mean={[f'{i:.04f}' for i in meant_overall_sbtn]}, std={[f'{i:.04f}' for i in stdt_overall_sbtn]}")
print(f"SR: mean={[f'{i:.04f}' for i in means_overall_sbtn]}, std={[f'{i:.04f}' for i in stds_overall_sbtn]}")

## class proportions

In [ ]:
fp_lmdb = os.path.join(fd_lmdb, 'sbtn.lmdb')
labels = []
env = lmdb.open(fp_lmdb, max_readers=1, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    n_items = txn.stat()['entries']
    keys = pickle.loads(txn.get('keys'.encode()))
    for i in range(int((n_items-1)//3)):
        lbl_obj = pickle.loads(txn.get((f'{i}_lc').encode()))
        lbl = np.frombuffer(lbl_obj[0], dtype=np.uint8).reshape(lbl_obj[1]).squeeze()
        labels.append(lbl)
labels = np.dstack(labels)
uni_vs = np.unique(labels)
print(n_items, keys, labels.shape, uni_vs)
for uni_v in uni_vs:
    na = np.sum(labels==uni_v)
    print(f'{uni_v}: {na}, {na/labels.size*100:.2f}%')

In [ ]:
72473/(264*264*22697)

original sbtn:
TOA: mean=[1.3973607e-01 1.2510014e-01 1.2389462e-01 1.3185014e-01 2.8322396e-01
 2.4637008e-01 1.7172308e-01 1.2479819e-01 2.4182030e-03 3.0288559e+02
 3.0148013e+02], std=[4.83666174e-02 5.44560663e-02 6.74990192e-02 9.90959033e-02
 9.89683494e-02 1.30543843e-01 1.24001555e-01 7.78222680e-02
 4.73055523e-03 1.08487473e+01 1.04910679e+01]
SR: mean=[ 9129.848  9585.586 10973.983 11667.794 17489.87  16538.992 13954.958], std=[2375.867  2545.3613 2975.5752 4044.4946 3713.2583 4925.824  4821.4604]

TOA: mean=[1.3031135e-01 1.1548932e-01 1.0973417e-01 1.1077589e-01 2.6778254e-01
 2.3235151e-01 1.5616804e-01 1.0803408e-01 2.0008080e-03 3.0310034e+02
 3.0169684e+02], std=[3.30185518e-02 3.84677909e-02 4.67350185e-02 6.74401373e-02
 8.56838375e-02 1.00014724e-01 9.49275419e-02 5.27144931e-02
 2.50696135e-03 9.69253922e+00 9.27867222e+00]
SR: mean=[ 8931.157  9326.412 10448.886 10865.018 16921.568 16003.465 13326.068], std=[1630.9384 1805.5847 2083.4126 2789.4658 3211.4546 3770.6658 3670.7993]

In [17]:
meant_overall_sbtn = meant.mean(axis=0)
stdt_overall_sbtn = np.sqrt(np.mean(stdt**2 + meant**2, axis=0) - meant_overall_sbtn**2)
means_overall_sbtn = means.mean(axis=0)
stds_overall_sbtn = np.sqrt(np.mean(stds**2 + means**2, axis=0) - means_overall_sbtn**2)

In [ ]:
for i, (nlcd, usfs, sbtn) in enumerate(zip(meant_overall_nlcd, meant_overall_usfs, meant_overall_sbtn)):
    print(f'{i}:{nlcd:.4f}, {usfs:.4f}, {sbtn:.4f}')
print('------------------------------------')
for i, (nlcd, usfs, sbtn) in enumerate(zip(stdt_overall_nlcd, stdt_overall_usfs, stdt_overall_sbtn)):
    print(f'{i}:{nlcd:.4f}, {usfs:.4f}, {sbtn:.4f}')
print('------------------------------------')
for i, (nlcd, usfs, sbtn) in enumerate(zip(means_overall_nlcd, means_overall_usfs, means_overall_sbtn)):
    print(f'{i}:{nlcd:.4f}, {usfs:.4f}, {sbtn:.4f}')
print('------------------------------------')
for i, (nlcd, usfs, sbtn) in enumerate(zip(stds_overall_nlcd, stds_overall_usfs, stds_overall_sbtn)):
    print(f'{i}:{nlcd:.4f}, {usfs:.4f}, {sbtn:.4f}')

In [ ]:
import cv2
# fp = '/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/Results_1st_stage_sub_s4/FT_dofab_auxl_attcp_exact_sub_max1500_deI_mcloss/0F_1D_10C_lr1e4_cos_w5_cos95/seed_42_bs_6/wandb/offline-run-20241021_092157-n2h3qi6d/files/media/images/mask/val_masks_0_341c695bd00f3881de54.mask.png'
# fp = '/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/Results_1st_stage_sub_s4/FT_dofab_auxl_attcp_exact_sub_max1500_deI_mcloss/0F_1D_10C_lr1e4_cos_w5_cos95/seed_42_bs_6/wandb/offline-run-20241021_092157-n2h3qi6d/files/media/images/mask/val_masks_0_fbcf450395ee65a3e998.mask.png'
fp = '/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/Results_1st_stage_sub_s4/FT_dofab_auxl_attcp_exact_sub_max1500_deI_mcloss/0F_1D_10C_lr1e4_cos_w5_cos95/seed_42_bs_6/wandb/offline-run-20241021_092157-n2h3qi6d/files/media/images/mask/val_masks_48_a6606e96d301aba08d40.mask.png'
img = cv2.imread(fp, cv2.IMREAD_UNCHANGED)
plt.imshow(img)
img.shape

In [ ]:
fp = '/p/project1/hai_ws4eo/liu_ch/Weak_FT/segment-anything-2/Results_1st_stage_sub_s4/FT_dofab_auxl_attcp_exact_sub_max1500_deI_mcloss/0F_5D_5C_lr1e4_cos_w5_cos95/seed_42_bs_6/wandb/offline-run-20241021_092146-tpme7zu1/files/media/images/mask/val_masks_48_8e9d6fabfbc979b3da95.mask.png'
img = cv2.imread(fp, cv2.IMREAD_UNCHANGED)
plt.imshow(img)
img.shape